# Fatigue Detection - PyTorch CNN (Cached Images)

**Run cells in order. Caching happens once to speed up training.**

In [1]:
# 1. Imports
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
# 2. Load parquet data
DATA_DIR = Path("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/closed-open-eyes/data")

parquet_files = sorted(DATA_DIR.rglob("*.parquet"))
df = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)

print(df.shape)
print(df.head())

In [ ]:
# 3. CACHE IMAGES (RUN ONLY ONCE - makes training 10x faster)
CACHE_DIR = Path("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/eye_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for i, row in df.iterrows():
    img = Image.open(io.BytesIO(row["Image_data"]["file"])).convert("RGB")
    img = img.resize((224, 224))
    out_path = CACHE_DIR / f"{i}.jpg"
    img.save(out_path, quality=95)
    rows.append({"path": str(out_path), "Label": row["Label"]})

cached_df = pd.DataFrame(rows)
cached_df.to_csv(CACHE_DIR / "eye_cache.csv", index=False)

print(cached_df.shape)
print(cached_df.head())

In [ ]:
# 4. Label map & test cached image
label_map = {"closed_eyes": 0, "open_eyes": 1}

sample = cached_df.iloc[0]
img = Image.open(sample["path"]).convert("RGB")
print(img.size)
plt.imshow(img)
plt.title(sample["Label"])
plt.axis("off")
plt.show()

In [ ]:
# 5. Cached dataset class
class EyeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = np.array(img, dtype=np.float32) / 255.0
        img = torch.tensor(img).permute(2, 0, 1)
        label = torch.tensor(label_map[row["Label"]], dtype=torch.long)
        return img, label

In [ ]:
# 6. Split & create loaders
train_df, val_df = train_test_split(
    cached_df,
    test_size=0.2,
    random_state=42,
    stratify=cached_df["Label"]
)

train_ds = EyeDataset(train_df)
val_ds = EyeDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

In [ ]:
# 7. CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# 8. Device & training setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# 9. Train loop
for epoch in range(5):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    print(f"Epoch {epoch+1} | loss={running_loss:.4f} | acc={correct/total:.4f}")

In [ ]:
# 10. Validate
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"Validation accuracy: {correct/total:.4f}")